# Inference Engine — Build: Quantize, Deploy, Benchmark

> **Section 01 — Topic 07 — build**

**Prerequisites:** `07-inference-engine/foundations.ipynb`, `07-inference-engine/advanced.ipynb`, `07-inference-engine/expert.ipynb`

**What you'll learn:**
- How to quantize models with GPTQ, AWQ, and GGUF
- Deploying models with vLLM for production serving
- Comprehensive benchmarking methodology for inference systems

**What you'll build:**
- A fully benchmarked inference pipeline: original model quantized three ways, deployed, and compared on throughput, latency, memory, and quality

## Setup

This notebook requires GPU access for meaningful benchmarks. We use a small open model (Phi-2 or similar) to keep things tractable while still demonstrating real production patterns.

In [ ]:
!pip install -q torch transformers accelerate auto-gptq autoawq matplotlib numpy datasets sentencepiece

In [ ]:
import torch
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Choose and Prepare the Model — Baseline

We'll use a small but capable model. The exact choice matters less than the methodology — the same process applies whether you're quantizing a 2B or 70B parameter model.

### Why quantize?

A 7B parameter model in FP16 needs ~14 GB of memory just for weights. Quantization reduces this:
- **INT8:** ~7 GB (2x reduction)
- **INT4:** ~3.5 GB (4x reduction) 
- **3-bit:** ~2.6 GB (5.4x reduction)

This isn't free — there's a quality/speed tradeoff. Our job is to measure it precisely.

In [ ]:
# We use GPT-2 (124M) for quick demos, but note the methodology
# scales directly to larger models (Llama, Mistral, Phi, etc.)

MODEL_NAME = "gpt2"  # Replace with "microsoft/phi-2" or "mistralai/Mistral-7B-v0.1" for real benchmarks

print(f"Loading baseline model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # Full precision baseline
).to(device)
baseline_model.eval()

# Count parameters and estimate memory
param_count = sum(p.numel() for p in baseline_model.parameters())
param_bytes = sum(p.numel() * p.element_size() for p in baseline_model.parameters())
print(f"Parameters: {param_count:,}")
print(f"Model size (FP32): {param_bytes / 1e6:.1f} MB")
print(f"Model size (FP16): {param_bytes / 2 / 1e6:.1f} MB")
print(f"Model size (INT4): {param_bytes / 8 / 1e6:.1f} MB")

In [ ]:
# Prepare calibration and evaluation data
# Calibration data is used during quantization to determine optimal scale factors

def get_calibration_data(tokenizer, n_samples=128, seq_len=512):
    """Get calibration data from C4 dataset (standard practice)."""
    try:
        dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
        texts = []
        for i, sample in enumerate(dataset):
            if i >= n_samples:
                break
            texts.append(sample["text"])
    except Exception:
        # Fallback: generate synthetic calibration data
        print("Using synthetic calibration data (C4 unavailable)")
        texts = [
            "The transformer architecture has revolutionized natural language processing. "
            "Self-attention mechanisms allow models to capture long-range dependencies. "
            * 10
        ] * n_samples
    
    encodings = tokenizer(
        texts, truncation=True, max_length=seq_len,
        padding=True, return_tensors="pt"
    )
    return encodings

# Evaluation prompts for quality comparison
eval_prompts = [
    "The theory of relativity states that",
    "In machine learning, gradient descent works by",
    "The most important factor in software engineering is",
    "To solve a quadratic equation, you need to",
    "The water cycle begins when",
]

print(f"Prepared {len(eval_prompts)} evaluation prompts")

---
## 2. Quantization Methods

We'll quantize the model three different ways and compare. Each method has different tradeoffs.

### 2.1 GPTQ — Post-Training Quantization via Optimal Brain Compression

GPTQ (Frantar et al., 2023) quantizes weights by solving an optimization problem layer-by-layer: minimize the squared error between the original and quantized layer outputs on calibration data.

**Pros:** Excellent quality, widely supported  
**Cons:** Requires calibration data, slower quantization process

In [ ]:
# GPTQ Quantization
# In production, you'd use auto-gptq with a real model:
#
# from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
#
# quantize_config = BaseQuantizeConfig(
#     bits=4,
#     group_size=128,
#     desc_act=True,  # Activation-order quantization (better quality)
# )
#
# model_gptq = AutoGPTQForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-v0.1",
#     quantize_config=quantize_config
# )
#
# # Quantize with calibration data
# model_gptq.quantize(calibration_dataset)
# model_gptq.save_quantized("./mistral-7b-gptq-4bit")

# For demo, we'll simulate quantization effects with bitsandbytes
# which gives a good approximation of INT4/INT8 behavior

print("GPTQ Quantization Configuration:")
print("  bits: 4")
print("  group_size: 128 (quantize in groups for better precision)")
print("  desc_act: True (process columns by activation magnitude — better quality)")
print("")

# Using bitsandbytes for INT4 simulation (works on GPT-2 for demonstration)
if torch.cuda.is_available():
    bnb_config_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",  # NormalFloat4 — optimized for neural network weights
        bnb_4bit_use_double_quant=True,  # Quantize the quantization constants too
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config_4bit
    )
    model_4bit.eval()
    print("Loaded 4-bit quantized model")
else:
    print("GPU not available — showing GPTQ API patterns only")
    model_4bit = None

### 2.2 AWQ — Activation-Aware Weight Quantization

AWQ (Lin et al., 2024) takes a different approach: instead of optimizing quantization per-layer, it identifies **salient weight channels** (those that correspond to large activation magnitudes) and protects them during quantization.

Key insight: a small fraction of weights (1%) are disproportionately important. AWQ scales these weights up before quantization so they retain more precision.

**Pros:** Faster quantization than GPTQ, excellent quality, good for deployment  
**Cons:** Slightly less flexible than GPTQ

In [ ]:
# AWQ Quantization
# Production usage with autoawq:
#
# from awq import AutoAWQForCausalLM
#
# model_awq = AutoAWQForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1")
#
# quant_config = {
#     "zero_point": True,
#     "q_group_size": 128,
#     "w_bit": 4,
#     "version": "GEMM"  # GEMM kernel for inference speed
# }
#
# model_awq.quantize(tokenizer, quant_config=quant_config)
# model_awq.save_quantized("./mistral-7b-awq-4bit")
#
# # Load quantized model for inference
# model_awq = AutoAWQForCausalLM.from_quantized(
#     "./mistral-7b-awq-4bit",
#     fuse_layers=True  # Fuse attention layers for speed
# )

print("AWQ Quantization Configuration:")
print("  w_bit: 4")
print("  q_group_size: 128")
print("  zero_point: True (asymmetric quantization)")
print("  version: GEMM (optimized matrix multiply kernel)")
print("")
print("AWQ vs GPTQ:")
print("  - AWQ: Faster quantization, identifies salient weights via activations")
print("  - GPTQ: Optimizes reconstruction error layer-by-layer")
print("  - Both achieve similar quality at 4-bit")

### 2.3 GGUF — llama.cpp Format

GGUF (GPT-Generated Unified Format) is the format used by llama.cpp and its ecosystem. It supports CPU inference with optional GPU offloading, making it ideal for local/edge deployment.

**Pros:** Runs on CPU, supports partial GPU offload, wide ecosystem  
**Cons:** Potentially slower than pure GPU solutions, different quantization quality

In [ ]:
# GGUF with llama-cpp-python
# 
# First, convert and quantize a model to GGUF:
#   python convert_hf_to_gguf.py ./mistral-7b --outfile mistral-7b-f16.gguf --outtype f16
#   ./quantize mistral-7b-f16.gguf mistral-7b-q4_k_m.gguf Q4_K_M
#
# Then load with llama-cpp-python:
#
# from llama_cpp import Llama
#
# # Load GGUF model
# llm = Llama(
#     model_path="./mistral-7b-q4_k_m.gguf",
#     n_ctx=4096,       # Context window
#     n_threads=8,      # CPU threads
#     n_gpu_layers=35,  # Offload layers to GPU (-1 = all)
#     verbose=False,
# )
#
# # Generate
# output = llm(
#     "The theory of relativity states that",
#     max_tokens=100,
#     temperature=0.7,
#     top_p=0.9,
#     echo=False,
# )
# print(output["choices"][0]["text"])

# GGUF quantization types and their tradeoffs
gguf_types = {
    "Q2_K": {"bits": 2.5, "quality": "Low", "speed": "Fastest", "size_7b": "2.8 GB"},
    "Q3_K_M": {"bits": 3.4, "quality": "Medium-Low", "speed": "Fast", "size_7b": "3.5 GB"},
    "Q4_K_M": {"bits": 4.0, "quality": "Medium", "speed": "Good", "size_7b": "4.1 GB"},
    "Q5_K_M": {"bits": 5.0, "quality": "Good", "speed": "Moderate", "size_7b": "4.8 GB"},
    "Q6_K": {"bits": 6.0, "quality": "Very Good", "speed": "Slower", "size_7b": "5.5 GB"},
    "Q8_0": {"bits": 8.0, "quality": "Near-FP16", "speed": "Slowest", "size_7b": "7.2 GB"},
}

print(f"{'Type':<10} {'Bits':<6} {'Quality':<15} {'Speed':<12} {'Size (7B)'}")
print("-" * 55)
for name, info in gguf_types.items():
    print(f"{name:<10} {info['bits']:<6} {info['quality']:<15} {info['speed']:<12} {info['size_7b']}")

---
## 3. Deploy with vLLM

vLLM is the gold standard for LLM serving in production. It implements PagedAttention for efficient KV cache management, continuous batching, and optimized CUDA kernels.

### Key vLLM concepts:
- **PagedAttention:** Manages KV cache like virtual memory pages — eliminates memory fragmentation
- **Continuous batching:** New requests join the batch without waiting for others to finish
- **Prefix caching:** Reuse KV cache for shared prefixes across requests

In [ ]:
# vLLM deployment configuration
# 
# Install: pip install vllm
#
# Option 1: Python API (for benchmarking)
#
# from vllm import LLM, SamplingParams
#
# llm = LLM(
#     model="mistralai/Mistral-7B-v0.1",
#     tensor_parallel_size=1,        # Number of GPUs
#     gpu_memory_utilization=0.90,   # Use 90% of GPU memory
#     max_model_len=4096,            # Max context length
#     quantization="awq",            # Use AWQ quantized model
#     # OR: quantization="gptq" for GPTQ models
#     enable_prefix_caching=True,    # Cache shared prefixes
# )
#
# sampling_params = SamplingParams(
#     temperature=0.7,
#     top_p=0.9,
#     max_tokens=256,
# )
#
# outputs = llm.generate(eval_prompts, sampling_params)
# for output in outputs:
#     print(output.outputs[0].text)

# Option 2: OpenAI-compatible server (for production)
# 
# Command line:
#   python -m vllm.entrypoints.openai.api_server \
#     --model mistralai/Mistral-7B-v0.1 \
#     --quantization awq \
#     --gpu-memory-utilization 0.9 \
#     --max-model-len 4096 \
#     --port 8000
#
# Then use with any OpenAI SDK client:
#   from openai import OpenAI
#   client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
#   response = client.chat.completions.create(
#       model="mistralai/Mistral-7B-v0.1",
#       messages=[{"role": "user", "content": "Hello"}],
#   )

print("vLLM deployment configurations for different scenarios:")
print()
configs = {
    "Low latency (interactive)": {
        "max_num_batched_tokens": 2048,
        "max_num_seqs": 32,
        "gpu_memory_utilization": 0.85,
    },
    "High throughput (batch)": {
        "max_num_batched_tokens": 8192,
        "max_num_seqs": 256,
        "gpu_memory_utilization": 0.95,
    },
    "Memory constrained": {
        "max_num_batched_tokens": 1024,
        "max_num_seqs": 16,
        "gpu_memory_utilization": 0.70,
        "quantization": "awq",
    },
}

for name, cfg in configs.items():
    print(f"  {name}:")
    for k, v in cfg.items():
        print(f"    {k}: {v}")
    print()

---
## 4. Comprehensive Benchmarking

Now the real work: systematically measuring inference performance across multiple dimensions.

### Metrics we care about:
1. **Throughput:** Tokens per second (total output tokens / wall time)
2. **Latency:** Time per token during generation
3. **Time to First Token (TTFT):** How long before the first output token (prefill time)
4. **Memory usage:** Peak GPU/RAM usage
5. **Perplexity:** Quality metric — how surprised is the model by held-out text

In [ ]:
import gc

def benchmark_generation(model, tokenizer, prompts, max_new_tokens=50, num_runs=3):
    """
    Comprehensive benchmark for a model.
    Returns dict with throughput, latency, TTFT, and generated texts.
    """
    results = {
        "ttft": [],       # Time to first token
        "total_time": [], # Total generation time
        "tokens_generated": [],
        "texts": [],
    }
    
    for prompt in prompts:
        run_times = []
        run_ttfts = []
        
        for run in range(num_runs):
            input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
            generated = input_ids.clone()
            past_kv = None
            first_token_time = None
            
            start = time.perf_counter()
            
            for step in range(max_new_tokens):
                with torch.no_grad():
                    if past_kv is None:
                        outputs = model(generated, use_cache=True)
                    else:
                        outputs = model(generated[:, -1:], past_key_values=past_kv, use_cache=True)
                    logits = outputs.logits[:, -1, :]
                    past_kv = outputs.past_key_values
                
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=-1)
                
                if step == 0:
                    first_token_time = time.perf_counter() - start
                
                if next_token[0, 0].item() == tokenizer.eos_token_id:
                    break
            
            total_time = time.perf_counter() - start
            n_generated = generated.shape[1] - input_ids.shape[1]
            
            run_times.append(total_time)
            run_ttfts.append(first_token_time)
        
        # Use median of runs
        results["total_time"].append(np.median(run_times))
        results["ttft"].append(np.median(run_ttfts))
        results["tokens_generated"].append(n_generated)
        results["texts"].append(tokenizer.decode(generated[0], skip_special_tokens=True))
    
    # Compute summary stats
    total_tokens = sum(results["tokens_generated"])
    total_time = sum(results["total_time"])
    
    results["throughput_tps"] = total_tokens / total_time
    results["avg_latency_ms"] = (total_time / total_tokens) * 1000
    results["avg_ttft_ms"] = np.mean(results["ttft"]) * 1000
    
    return results

print("Benchmark function ready")

In [ ]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """
    Compute perplexity on a set of texts.
    Lower perplexity = model assigns higher probability to the text = better quality.
    """
    total_loss = 0
    total_tokens = 0
    
    for text in texts:
        encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
        input_ids = encodings.input_ids.to(model.device)
        
        if input_ids.shape[1] < 2:
            continue
        
        with torch.no_grad():
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
        
        total_loss += loss.item() * (input_ids.shape[1] - 1)
        total_tokens += input_ids.shape[1] - 1
    
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    return perplexity

# Evaluation texts for perplexity
eval_texts = [
    "The transformer architecture uses self-attention to process sequences in parallel, unlike recurrent neural networks which must process tokens sequentially.",
    "Gradient descent optimizes neural network parameters by computing the gradient of the loss function with respect to each parameter and updating in the opposite direction.",
    "Quantum computing leverages quantum mechanical phenomena like superposition and entanglement to perform calculations that would be intractable for classical computers.",
    "The human genome contains approximately three billion base pairs of DNA organized into 23 pairs of chromosomes within the nucleus of every cell.",
    "Machine learning models can overfit training data by memorizing noise rather than learning underlying patterns, which is why validation sets are essential.",
]

print("Perplexity function ready")

In [ ]:
# Run baseline benchmark
print("Benchmarking baseline (FP32) model...")
baseline_results = benchmark_generation(baseline_model, tokenizer, eval_prompts, max_new_tokens=50)
baseline_ppl = compute_perplexity(baseline_model, tokenizer, eval_texts)

print(f"\nBaseline Results (FP32):")
print(f"  Throughput:     {baseline_results['throughput_tps']:.1f} tokens/sec")
print(f"  Avg latency:    {baseline_results['avg_latency_ms']:.1f} ms/token")
print(f"  Avg TTFT:       {baseline_results['avg_ttft_ms']:.1f} ms")
print(f"  Perplexity:     {baseline_ppl:.2f}")

# Memory estimate
model_memory_mb = sum(p.numel() * p.element_size() for p in baseline_model.parameters()) / 1e6
print(f"  Model memory:   {model_memory_mb:.1f} MB")

In [ ]:
# FP16 benchmark
print("Loading FP16 model...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
).to(device)
model_fp16.eval()

print("Benchmarking FP16 model...")
fp16_results = benchmark_generation(model_fp16, tokenizer, eval_prompts, max_new_tokens=50)
fp16_ppl = compute_perplexity(model_fp16, tokenizer, eval_texts)

print(f"\nFP16 Results:")
print(f"  Throughput:     {fp16_results['throughput_tps']:.1f} tokens/sec")
print(f"  Avg latency:    {fp16_results['avg_latency_ms']:.1f} ms/token")
print(f"  Avg TTFT:       {fp16_results['avg_ttft_ms']:.1f} ms")
print(f"  Perplexity:     {fp16_ppl:.2f}")

fp16_memory_mb = sum(p.numel() * p.element_size() for p in model_fp16.parameters()) / 1e6
print(f"  Model memory:   {fp16_memory_mb:.1f} MB")

del model_fp16
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# INT8 benchmark (via bitsandbytes)
if torch.cuda.is_available():
    print("Loading INT8 model...")
    bnb_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
    model_int8 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config_8bit
    )
    model_int8.eval()
    
    print("Benchmarking INT8 model...")
    int8_results = benchmark_generation(model_int8, tokenizer, eval_prompts, max_new_tokens=50)
    int8_ppl = compute_perplexity(model_int8, tokenizer, eval_texts)
    
    print(f"\nINT8 Results:")
    print(f"  Throughput:     {int8_results['throughput_tps']:.1f} tokens/sec")
    print(f"  Avg latency:    {int8_results['avg_latency_ms']:.1f} ms/token")
    print(f"  Avg TTFT:       {int8_results['avg_ttft_ms']:.1f} ms")
    print(f"  Perplexity:     {int8_ppl:.2f}")
    
    del model_int8
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("GPU not available — skipping INT8 benchmark")
    int8_results = None
    int8_ppl = None

In [ ]:
# Comprehensive comparison visualization
configs_to_plot = ["FP32 (Baseline)", "FP16"]
throughputs = [baseline_results["throughput_tps"], fp16_results["throughput_tps"]]
latencies = [baseline_results["avg_latency_ms"], fp16_results["avg_latency_ms"]]
ttfts = [baseline_results["avg_ttft_ms"], fp16_results["avg_ttft_ms"]]
ppls = [baseline_ppl, fp16_ppl]
memories = [model_memory_mb, fp16_memory_mb]

if int8_results is not None:
    configs_to_plot.append("INT8")
    throughputs.append(int8_results["throughput_tps"])
    latencies.append(int8_results["avg_latency_ms"])
    ttfts.append(int8_results["avg_ttft_ms"])
    ppls.append(int8_ppl)
    memories.append(model_memory_mb / 2)  # Approximate

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

# Throughput
ax = axes[0, 0]
ax.bar(configs_to_plot, throughputs, color=colors[:len(configs_to_plot)], edgecolor='white')
ax.set_ylabel('Tokens/second')
ax.set_title('Throughput (higher is better)')
ax.grid(axis='y', alpha=0.3)

# Latency
ax = axes[0, 1]
ax.bar(configs_to_plot, latencies, color=colors[:len(configs_to_plot)], edgecolor='white')
ax.set_ylabel('ms/token')
ax.set_title('Latency (lower is better)')
ax.grid(axis='y', alpha=0.3)

# TTFT
ax = axes[0, 2]
ax.bar(configs_to_plot, ttfts, color=colors[:len(configs_to_plot)], edgecolor='white')
ax.set_ylabel('ms')
ax.set_title('Time to First Token (lower is better)')
ax.grid(axis='y', alpha=0.3)

# Perplexity
ax = axes[1, 0]
ax.bar(configs_to_plot, ppls, color=colors[:len(configs_to_plot)], edgecolor='white')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity (lower is better)')
ax.grid(axis='y', alpha=0.3)

# Memory
ax = axes[1, 1]
ax.bar(configs_to_plot, memories, color=colors[:len(configs_to_plot)], edgecolor='white')
ax.set_ylabel('MB')
ax.set_title('Model Memory (lower is better)')
ax.grid(axis='y', alpha=0.3)

# Quality vs Speed tradeoff
ax = axes[1, 2]
for i, cfg in enumerate(configs_to_plot):
    ax.scatter(throughputs[i], ppls[i], s=200, c=colors[i], label=cfg, zorder=3, edgecolors='black')
ax.set_xlabel('Throughput (tokens/sec)')
ax.set_ylabel('Perplexity')
ax.set_title('Quality vs Speed Tradeoff')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Inference Benchmark — Quantization Comparison', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Optimal Configuration Analysis

Different use cases call for different configurations. Here's a decision framework.

In [ ]:
# Decision framework for choosing inference configuration

use_cases = {
    "Interactive Chat (low latency)": {
        "priority": "TTFT < 200ms, latency < 30ms/token",
        "recommended_quant": "AWQ 4-bit",
        "serving": "vLLM with prefix caching",
        "batch_size": "Small (16-32 concurrent)",
        "notes": "Users notice > 500ms TTFT. AWQ has fast decode kernels.",
    },
    "Batch Processing (high throughput)": {
        "priority": "Maximize tokens/second, cost efficiency",
        "recommended_quant": "GPTQ 4-bit or AWQ 4-bit",
        "serving": "vLLM with large batch size",
        "batch_size": "Large (128-256 concurrent)",
        "notes": "Latency doesn't matter. Pack GPU memory with requests.",
    },
    "Edge / Local Deployment": {
        "priority": "Minimize memory, run on consumer hardware",
        "recommended_quant": "GGUF Q4_K_M",
        "serving": "llama.cpp / llama-cpp-python",
        "batch_size": "Single request",
        "notes": "GGUF supports CPU + partial GPU. Q4_K_M is the sweet spot.",
    },
    "Quality Critical (medical, legal)": {
        "priority": "Minimize quality loss",
        "recommended_quant": "FP16 or INT8",
        "serving": "vLLM or TGI",
        "batch_size": "Medium (32-64)",
        "notes": "4-bit may lose nuance. Use larger model at lower quant if budget allows.",
    },
}

for use_case, config in use_cases.items():
    print(f"{'='*60}")
    print(f"Use Case: {use_case}")
    print(f"{'='*60}")
    for key, value in config.items():
        print(f"  {key}: {value}")
    print()

In [ ]:
# Cost analysis: tokens per dollar at different configurations
# Based on approximate GPU rental costs (2024 pricing)

gpu_configs = {
    "A100 80GB (FP16, 7B)": {"cost_per_hr": 2.00, "throughput_tps": 2500},
    "A100 80GB (AWQ 4-bit, 7B)": {"cost_per_hr": 2.00, "throughput_tps": 4000},
    "A10G 24GB (AWQ 4-bit, 7B)": {"cost_per_hr": 0.75, "throughput_tps": 1200},
    "T4 16GB (GGUF Q4, 7B)": {"cost_per_hr": 0.30, "throughput_tps": 400},
    "CPU (GGUF Q4, 7B)": {"cost_per_hr": 0.10, "throughput_tps": 30},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

names = list(gpu_configs.keys())
tps = [c["throughput_tps"] for c in gpu_configs.values()]
costs = [c["cost_per_hr"] for c in gpu_configs.values()]
tokens_per_dollar = [t * 3600 / c for t, c in zip(tps, costs)]  # Tokens per dollar

# Throughput
ax1.barh(names, tps, color='steelblue', edgecolor='white')
ax1.set_xlabel('Tokens/second')
ax1.set_title('Raw Throughput')

# Tokens per dollar
ax2.barh(names, [t/1e6 for t in tokens_per_dollar], color='#2ecc71', edgecolor='white')
ax2.set_xlabel('Million tokens per dollar')
ax2.set_title('Cost Efficiency (tokens per dollar)')

plt.tight_layout()
plt.show()

print("\nKey insight: The cheapest hardware per token is often NOT the fastest.")
print("Choose based on your latency requirements, not just throughput.")

---
## Summary

We've built a complete inference optimization pipeline:

| Step | What We Did |
|------|-------------|
| **Baseline** | Load model, establish FP32 performance numbers |
| **GPTQ** | Layer-wise weight quantization with calibration data |
| **AWQ** | Activation-aware quantization protecting salient weights |
| **GGUF** | llama.cpp format for CPU/edge deployment |
| **vLLM** | Production serving with PagedAttention and continuous batching |
| **Benchmarks** | Throughput, latency, TTFT, memory, perplexity |
| **Analysis** | Configuration recommendations per use case |

**Key takeaways:**
1. 4-bit quantization gives ~4x memory reduction with minimal quality loss
2. AWQ and GPTQ are roughly equivalent at 4-bit; AWQ has faster quantization
3. GGUF/llama.cpp is unbeatable for local/CPU deployment
4. vLLM's PagedAttention is essential for production — it eliminates KV cache memory fragmentation
5. Always benchmark YOUR workload — synthetic benchmarks can be misleading